# 🔬 TSRL — Strategy Backtesting

Run backtests with multiple strategies, compare performance, and visualize equity curves.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

from src.application.services.data_service import DataService
from src.engine.backtest.engine import BacktestEngine, BacktestConfig
from src.strategies.momentum.ema_crossover import EMACrossoverStrategy
from src.strategies.momentum.macd_strategy import MACDStrategy
from src.strategies.mean_reversion.bollinger_bands import BollingerBandsStrategy
from src.strategies.momentum.ma_ribbon import MovingAverageRibbonStrategy

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

## 1. Setup

In [ ]:
# Fetch data
data_service = DataService()
df, source = data_service.fetch_data('AAPL', datetime(2022, 1, 1), datetime(2024, 12, 31))
print(f'Loaded {len(df)} bars ({source})')

# Define strategies to test
strategies = {
    'EMA Crossover': EMACrossoverStrategy(fast_period=10, slow_period=30),
    'MACD': MACDStrategy(),
    'Bollinger Bands': BollingerBandsStrategy(),
    'MA Ribbon': MovingAverageRibbonStrategy(),
}

config = BacktestConfig(
    initial_capital=100000,
    commission=0.001,
    slippage=0.0005,
)

## 2. Run Backtests

In [ ]:
results = {}
for name, strategy in strategies.items():
    engine = BacktestEngine(config)
    result = engine.run(strategy, df)
    results[name] = result
    print(f'{name}: Return={result.total_return*100:.2f}%, Trades={len(result.trades)}, Sharpe={result.metrics.sharpe_ratio:.2f}')

## 3. Performance Comparison Table

In [ ]:
comparison = []
for name, result in results.items():
    m = result.metrics
    comparison.append({
        'Strategy': name,
        'Return': f'{result.total_return * 100:.2f}%',
        'Final Capital': f'${result.final_capital:,.0f}',
        'Trades': len(result.trades),
        'Sharpe': f'{m.sharpe_ratio:.2f}',
        'Sortino': f'{m.sortino_ratio:.2f}',
        'Max DD': f'{m.max_drawdown_pct:.2f}%',
        'Win Rate': f'{m.win_rate * 100:.1f}%',
        'Profit Factor': f'{m.profit_factor:.2f}',
    })

pd.DataFrame(comparison).set_index('Strategy')

## 4. Equity Curves

In [ ]:
fig, ax = plt.subplots()

colors = ['#4fc3f7', '#ff5252', '#69f0ae', '#ffd740']
for (name, result), color in zip(results.items(), colors):
    if result.equity_curve is not None and not result.equity_curve.empty:
        equity_col = 'equity' if 'equity' in result.equity_curve.columns else 'total'
        if equity_col in result.equity_curve.columns:
            ec = result.equity_curve[equity_col]
            ax.plot(ec.index, ec.values, label=name, color=color, linewidth=1.5)

ax.axhline(y=config.initial_capital, color='gray', linestyle='--', alpha=0.5, label='Initial Capital')
ax.set_title('Equity Curves — Strategy Comparison', fontsize=14, fontweight='bold')
ax.set_ylabel('Portfolio Value ($)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Drawdown Analysis

In [ ]:
fig, ax = plt.subplots()

for (name, result), color in zip(results.items(), colors):
    if result.equity_curve is not None and not result.equity_curve.empty:
        equity_col = 'equity' if 'equity' in result.equity_curve.columns else 'total'
        if equity_col in result.equity_curve.columns:
            ec = result.equity_curve[equity_col]
            running_max = ec.cummax()
            drawdown = (ec - running_max) / running_max * 100
            ax.fill_between(drawdown.index, drawdown.values, 0, alpha=0.3, color=color)
            ax.plot(drawdown.index, drawdown.values, label=name, color=color, linewidth=1)

ax.set_title('Drawdown Comparison', fontsize=14, fontweight='bold')
ax.set_ylabel('Drawdown (%)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Trade Distribution

In [ ]:
fig, axes = plt.subplots(1, len(strategies), figsize=(16, 5))
if len(strategies) == 1:
    axes = [axes]

for ax, (name, result), color in zip(axes, results.items(), colors):
    pnls = [t.pnl for t in result.trades if t.pnl is not None]
    if pnls:
        wins = [p for p in pnls if p > 0]
        losses = [p for p in pnls if p <= 0]
        ax.hist(wins, bins=20, alpha=0.7, color='#69f0ae', label=f'Wins ({len(wins)})')
        ax.hist(losses, bins=20, alpha=0.7, color='#ff5252', label=f'Losses ({len(losses)})')
    ax.set_title(name, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Trade P&L Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()